# CfC / BAOAB propagator — Anisotropic Gaussian V_θ + Fock-Reg — OpenWebText d=384

## Why this run exists

The `γ=0.10` and `γ=0.30` d=384 runs of
`colab_fock_aniso_gaussian_fockreg_openwebtext.ipynb` both hit chronic
gradient-spike instability under the damped velocity-Verlet integrator.
The `γ=0.10` arm eventually stalled outright: 13 watchdog reloads between
steps 10K and 17K, **zero** PPL improvement over 6,900 steps, and a
record pre-clip gradient of 263,084.

Two structural properties of the Verlet step cause this:

1. **The stiff part of the force is integrated explicitly.** A token in a
   sharp V_θ well has large local curvature `K`; the explicit update is
   stable only while `dt < 2·sqrt(m/K)`. As wells sharpen during training
   the layer step silently crosses that bound and the state amplifies
   geometrically down the remaining layers — which is what a gradient
   spike looks like from outside.
2. **V_θ sits inside the second-order `create_graph` chain**, because the
   force comes from `autograd.grad(V_θ + V_φ, create_graph=True)`.

This notebook runs the fix, in attributable stages, via a single
`INTEGRATOR` switch:

| `INTEGRATOR` | V_θ force | Integrator | What it isolates |
|---|---|---|---|
| `'verlet'` | autograd | damped velocity-Verlet | the existing baseline, bit-identical to the runs above |
| `'analytic_vtheta'` | **closed form** | damped velocity-Verlet | how much of the spiking is the V_θ half of the `create_graph` cascade |
| `'baoab'` | closed form | ABOBA split, exact `exp(-γdt)` friction | the integrator split alone, no CfC |
| `'baoab_cfc'` | closed form | ABOBA split + **closed-form harmonic propagator** for the stiff part of V_θ | the full fix |

Each arm writes to its own Drive folder (the integrator is part of the
variant tag), so arms can be run one at a time and resumed independently.

## What is guaranteed, and by what

- **`'analytic_vtheta'` is the same model, not a different one.**
  `test_cfc_baoab.py::test_analytic_vtheta_equivalence` asserts that
  switching V_θ's force to its closed form leaves the loss and *every
  parameter gradient* unchanged (worst relative error ~2e-6).
- **CfC changes how the force is integrated, not what the force is.**
  The stiff diagonal part of V_θ is propagated by its exact harmonic
  solution and the residual is kicked numerically; the two sum to the
  unmodified total force.
  `test_cfc_baoab.py::test_cfc_force_preservation` verifies the CfC and
  plain-BAOAB steps agree to O(dt³) — a second-order discrepancy would
  mean the force field had changed.
- **The propagator cannot blow up.** All Gaussian wells are attractive, so
  the stiffness is non-negative and the substep is always a bounded
  rotation in phase space. At `K = 10⁴` (ω·dt = 100), twelve explicit
  steps overflow float32 while the CfC step stays inside its initial
  orbit.

Theory: `companion_notes/Closed_Form_and_Hybrid_Integration_Strategies_for_Fock-PARFLM.md`,
`companion_notes/Blended_CfC_BAOAB_Deep_Dive.md`, and `paper_v5` §20.
Implementation: `parf/cfc_baoab.py` (propagator), `parf/model_parf_multixi.py`
(`_layer_step_langevin`), `parf/model_aniso_gaussian_vtheta.py`
(`harmonic_terms`).

## Everything else is held fixed

Same d=384 / L=16 / M=32 architecture, same 5-channel ξ, same anisotropic
depth-conditioned V_θ (5 heads × 8 wells, rank 4), same Fock coupling
regularisation, same WSD schedule, same per-group gradient clips, same
watchdog. Only the integrator changes.


In [ ]:
# == Cell 0: Configuration =============================================

# -- V_theta: Anisotropic Gaussian (diagonal + low-rank precision) -----
V_THETA_VARIANT             = 'aniso_gaussian'
V_THETA_WELLS_PER_HEAD      = 8
V_THETA_DEPTH_CONDITION     = True
V_THETA_DEPTH_CODE_INIT_STD = 0.02
ANISO_RANK                  = 4
W_SCALE                     = 1.0

# -- Xi channels (5long preset from OWT notebook) ----------------------
XI_OVERRIDE     = '5long'
_XI_PRESETS_CFG = {
    5:       [0.25, 0.50, 0.75, 0.95, 0.99],
    '5long': [0.50, 0.75, 0.95, 0.99, 0.995],
    6:       [0.25, 0.50, 0.75, 0.95, 0.99, 0.995],
    '4long': [0.50, 0.75, 0.95, 0.995],
}
XI_ALPHA_INITS = _XI_PRESETS_CFG[XI_OVERRIDE]
XI_CHANNELS    = len(XI_ALPHA_INITS)
V_THETA_N_HEADS = XI_CHANNELS

# -- PARF V_phi --------------------------------------------------------
V_PHI_KIND      = 'structural_competitive'
V_PHI_MLP_HIDDEN = 128
TOP_K           = 16
V_PHI_N_HEADS   = 4
V_PHI_D_TYPE    = 32
V_PHI_D_ANGLE   = 16

# -- Reverse channel stabilisation (E5c) -------------------------------
REVERSE_CHANNEL              = True
REVERSE_CHANNEL_STABLE       = True
REVERSE_CHANNEL_PRE_LN       = True
REVERSE_CHANNEL_SOFT_NORM    = True
REVERSE_CHANNEL_WARMUP_STEPS = 4000
REVERSE_CHANNEL_PER_LAYER    = True
REVERSE_CHANNEL_RESET_SCALE  = False

# -- Register repulsion (B4) -------------------------------------------
REGISTER_REPULSION       = True
REGISTER_REPULSION_COEFF = 0.05
REGISTER_REPULSION_KIND  = 'gram'

# -- Output head -------------------------------------------------------
USE_OUTPUT_BIAS = True
TIE_EMBEDDINGS  = False

# -- Optimizer ---------------------------------------------------------
OPTIMIZER = 'adamw'
GRAD_CENTRALIZATION = False

# -- LR schedule (WSD) -------------------------------------------------
LR_SCHEDULE     = 'wsd'
WSD_WARMUP_FRAC = 0.05
WSD_STABLE_FRAC = 0.60
WSD_LR_FLOOR    = None          # resolved after LR is set

# -- Batch / accumulation ----------------------------------------------
# The CfC arm carries a second anisotropic-well evaluation per layer
# (harmonic_terms at h, plus the force at the drifted h_mid), so its
# activation footprint is ~1.5x the Verlet arm's even with the well
# parameters shared between the two.  The auto-probe in Cell 5 therefore
# tends to land on a smaller per-device batch than the Verlet notebook
# does.  Leaving GRAD_ACCUM fixed would then shrink the *effective*
# batch too, which would confound a Verlet-vs-CfC comparison: the two
# runs would differ in gradient noise as well as in integrator.  So the
# probe compensates -- it keeps EFFECTIVE_BATCH pinned to the target and
# spends the difference on accumulation steps.
#
# 32 matches the Verlet aniso-Gaussian OWT run
# (colab_fock_aniso_gaussian_fockreg_openwebtext.ipynb on an 80GB card:
# batch 16 x accum 2), so PPL curves stay directly comparable.
TARGET_EFFECTIVE_BATCH = 32
GRAD_ACCUM      = 2       # fallback / lower bound; raised by the probe

# -- Fock coupling regularisation --------------------------------------
LAMBDA_FOCK_REG = 5e-3
FOCK_REG_EPS    = 1e-6

# == INTEGRATOR =========================================================
# 'verlet'          : damped velocity-Verlet, friction folded into the
#                     1/(1+dt*gamma) coefficient, V_theta force from
#                     autograd.  The historical baseline -- bit-identical
#                     to the runs this notebook is trying to improve on.
# 'analytic_vtheta' : same integrator, but -grad V_theta comes from its
#                     closed form, so V_theta leaves the second-order
#                     create_graph chain.  Same model, same gradients
#                     (asserted by test_cfc_baoab.py) -- only the way the
#                     force is obtained differs.
# 'baoab'           : palindromic ABOBA split with an exact exp(-gamma*dt)
#                     friction substep and a genuine velocity.
# 'baoab_cfc'       : as 'baoab', plus the closed-form harmonic propagator
#                     for the stiff diagonal part of V_theta.  Immune to
#                     the well-sharpening blow-up that the explicit step
#                     suffers from.
INTEGRATOR = 'baoab_cfc'

# O-step thermostat temperature.  0.0 = deterministic friction only, which
# keeps this run directly comparable to the Verlet curves.  Raising it
# turns the O-step into a true FDT-locked Langevin thermostat.
LANGEVIN_T = 0.0

_INTEGRATOR_MODES = {
    #                     cfg.integrator   cfg.vtheta_analytic_force
    'verlet':            ('verlet',        False),
    'analytic_vtheta':   ('verlet',        True),
    'baoab':             ('baoab',         True),
    'baoab_cfc':         ('baoab_cfc',     True),
}
assert INTEGRATOR in _INTEGRATOR_MODES, (
    f'INTEGRATOR={INTEGRATOR!r} not in {sorted(_INTEGRATOR_MODES)}')
CFG_INTEGRATOR, CFG_VTHETA_ANALYTIC = _INTEGRATOR_MODES[INTEGRATOR]

# -- Damping coefficient -------------------------------------------------
# gamma=0.100 chosen from the d=384, L=16 aniso-Gaussian+fock-reg gamma
# sweep (colab_fock_gamma_sweep_geodesic_aniso_gaussian_fockreg_d384.ipynb):
# best PPL (278.27) AND best geodesic R_bar (0.6708) coincide at gamma=0.100.
# gamma=0.150/0.250 are tied within ~5% (flat bowl); gamma=0.200 was an
# isolated instability outlier (PPL=2250) bracketed by good neighbours on
# both sides, not a genuine stability wall.
# Under BAOAB the friction is applied as exp(-gamma*dt) rather than
# 1/(1+gamma*dt); at gamma=0.10, dt=1 the two differ by ~0.5%, so the same
# gamma remains directly comparable across arms.
FIXED_GAMMA = 0.10

# -- Regularisation ----------------------------------------------------
LAMBDA_V       = 1e-2
BG_QUAD_EPS    = 0.0

# -- Training ----------------------------------------------------------
TOTAL_STEPS   = 100_000
BLOCK_SIZE    = 512
VOCAB_SIZE    = 50257
SEED          = 0

# -- Variant tag (for GDrive path and checkpoint naming) ----------------
_variant_parts = []
_variant_parts.append(f'xi{XI_OVERRIDE}')
_variant_parts.append(f'topk{TOP_K}')
_variant_parts.append(f'dt{V_PHI_D_TYPE}da{V_PHI_D_ANGLE}')
_variant_parts.append(f'mh{V_PHI_N_HEADS}')
_variant_parts.append(f'aniso_dcvt{V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}')
_variant_parts.append('ob')
_variant_parts.append('untied')
_variant_parts.append(LR_SCHEDULE)
_variant_parts.append('e5c')
_variant_parts.append('plgate')
_variant_parts.append(f'rep{REGISTER_REPULSION_COEFF:g}')
_variant_parts.append(f'fockreg{LAMBDA_FOCK_REG:g}')
_variant_parts.append(f'g{FIXED_GAMMA:g}')
# The integrator is part of the tag, so every arm gets its own Drive
# folder, checkpoints and training_log.jsonl and can be resumed on its own.
_variant_parts.append(INTEGRATOR)
if LANGEVIN_T > 0:
    _variant_parts.append(f'T{LANGEVIN_T:g}')
_variant_tag = '_'.join(_variant_parts)

total_wells = V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD
print(f'Anisotropic Gaussian V_theta on OpenWebText d=384')
print(f'  V_theta: {V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
      f'{total_wells} total attractors')
print(f'  Aniso rank r={ANISO_RANK}')
print(f'  Depth-conditioned: {V_THETA_DEPTH_CONDITION}')
print(f'  Fock coupling reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  Damping: fixed_gamma={FIXED_GAMMA}')
print(f'  Integrator: {INTEGRATOR}  '
      f'(cfg.integrator={CFG_INTEGRATOR}, '
      f'analytic_vtheta={CFG_VTHETA_ANALYTIC}, T={LANGEVIN_T:g})')
print(f'  Xi: {XI_CHANNELS}ch  horizons ~{[round(1/(1-a),1) for a in XI_ALPHA_INITS]} tok')
print(f'  V_phi={V_PHI_KIND} x {V_PHI_N_HEADS}h  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')
print(f'  steps={TOTAL_STEPS}  schedule={LR_SCHEDULE}  grad_accum={GRAD_ACCUM}')
print(f'  [variant] tag={_variant_tag}')

In [ ]:
# == Cell 1: Environment + Drive Mount =================================
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path
from dataclasses import asdict

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _gdrive_name = 'semsimula_fock_cfc_baoab_owt'
    if _variant_tag:
        _gdrive_name += f'_{_variant_tag}'
    GDRIVE_ROOT = Path(f'/content/drive/MyDrive/{_gdrive_name}')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)
    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    CKPT_DIR    = GDRIVE_ROOT / 'checkpoints'
    RESULTS_DIR = GDRIVE_ROOT / 'results'
    CKPT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow matplotlib')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    _local_phase = 'cfc_baoab_owt' + (f'_{_variant_tag}' if _variant_tag else '')
    CKPT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase / 'ckpts'
    RESULTS_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase
    for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

CKPT_PREFIX   = 'fock_cfc_owt' + (f'_{_variant_tag}' if _variant_tag else '')
CKPT_INTERVAL = 7_500
CKPT_STEPS    = list(range(CKPT_INTERVAL, TOTAL_STEPS + 1, CKPT_INTERVAL))

print(f'CKPT_DIR    = {CKPT_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'Steps: {TOTAL_STEPS:,}  checkpoints at: {CKPT_STEPS}')

In [ ]:
# == Cell 2: GPU + Checkpoint resolution ===============================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print('WARNING: No GPU detected. This notebook requires CUDA.')

resume_step = 0
resume_ckpt = None

for s in sorted(CKPT_STEPS, reverse=True):
    cand = CKPT_DIR / f'{CKPT_PREFIX}_step{s}.pt'
    if cand.exists():
        resume_ckpt = cand
        resume_step = s
        break

_best_candidates = []
_canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
if _canonical.exists():
    _best_candidates.append(_canonical)
for _f in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt')):
    _best_candidates.append(_f)

_best_path = None
_best_step_found = resume_step
for _cand in _best_candidates:
    try:
        _bd = torch.load(_cand, map_location='cpu', weights_only=False)
        _s = _bd.get('step', 0)
        _p = _bd.get('val_ppl', float('inf'))
        del _bd
        if _s > _best_step_found:
            _best_step_found = _s
            _best_path = _cand
            _best_ppl = _p
            print(f'  Found best candidate: {_cand.name} (step {_s:,}, PPL {_p:.2f})')
    except Exception as e:
        print(f'[warn] could not inspect {_cand.name}: {e}')

if _best_path is not None and _best_step_found > resume_step:
    print(f'Best checkpoint (step {_best_step_found:,}, PPL {_best_ppl:.2f}) is more recent '
          f'than latest periodic checkpoint (step {resume_step:,}) -- resuming from best.')
    resume_ckpt = _best_path
    resume_step = _best_step_found

if resume_ckpt is not None:
    print(f'Resuming from: {resume_ckpt.name}  (step {resume_step:,})')
    print(f'Remaining: {TOTAL_STEPS - resume_step:,} steps')
else:
    print('No checkpoint found -- training from scratch.')
    print(f'Total: {TOTAL_STEPS:,} steps  Checkpoints every {CKPT_INTERVAL:,}')

In [ ]:
# == Cell 3: Data loading (OpenWebText) ================================
from data_module import get_batch

MAX_TRAIN_TOKENS = 2_000_000_000
VAL_TOKENS       = 2_000_000
CHUNK_SIZE       = 50_000

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

for alt_name in [
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05',
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd',
    'semsimula_fock_structured_vtheta_owt_phase4',
    'semsimula_fock_gaussian_sarf_openwebtext_phase5',
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
    'semsimula_fock_multihead_openwebtext',
    'semsimula_fock_multicontext_vtheta_owt',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        import shutil
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens ...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

In [ ]:
# == Cell 4: V_theta + integrator: import from the repo, then verify ====
#
# The template notebook inlined AnisotropicMixtureGaussianVTheta.  This one
# imports it instead, because the CfC propagator needs `harmonic_terms`,
# which lives with the class in parf/model_aniso_gaussian_vtheta.py.  An
# inlined copy would silently shadow it and fall back to a stale definition.

from model_aniso_gaussian_vtheta import (
    AnisotropicMixtureGaussianVTheta,
    AnisotropicMultiContextGaussianVTheta,
    AnisotropicDepthConditionedGaussianVTheta,
    install_aniso_depth_routing,
)
import model_parf_multixi as _mpm

# -- Stale-checkout guards (fail here, not 90 minutes into training) -----
assert hasattr(AnisotropicDepthConditionedGaussianVTheta, 'harmonic_terms'), (
    'STALE CHECKOUT: the anisotropic V_theta has no harmonic_terms(), which '
    'the CfC propagator needs. Restart the Colab runtime (Runtime > Restart '
    'runtime) so the freshly fetched module is re-imported.')
assert hasattr(_mpm.MultiXiPARFLM, '_layer_step_langevin'), (
    'STALE CHECKOUT: MultiXiPARFLM has no _layer_step_langevin(). Restart '
    'the Colab runtime and re-run from the top.')
assert 'integrator' in {f.name for f in
                        __import__('dataclasses').fields(_mpm.MultiXiPARFConfig)}, (
    'STALE CHECKOUT: MultiXiPARFConfig has no `integrator` field.')

# -- Run the integrator test suite (CPU, ~10 s) --------------------------
# Cheap insurance: proves on THIS checkout that the analytic V_theta force
# reproduces autograd's gradients exactly, that the CfC split preserves the
# force field to O(dt^3), and that the propagator survives stiffness that
# overflows the explicit step.
import subprocess, sys as _sys
_test = CA_DIR / 'parf' / 'test_cfc_baoab.py'
if _test.exists():
    _res = subprocess.run([_sys.executable, str(_test)],
                          capture_output=True, text=True, cwd=str(CA_DIR / 'parf'))
    print(_res.stdout[-2500:])
    if _res.returncode != 0:
        print(_res.stderr[-2500:])
        raise RuntimeError('CfC/BAOAB integrator tests FAILED -- do not train '
                           'on this checkout.')
else:
    print(f'WARNING: {_test} not found; skipping integrator self-tests.')


In [ ]:
# == Cell 5: Model config + build + aniso V_theta swap =================
import math
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

# Guard against a stale in-memory module for the classes THIS RUN
# actually instantiates -- FockMultiXiPARFLM (model_fock_parf_multixi.py)
# on top of MultiXiPARFLM (model_parf_multixi.py). These have their OWN
# copies of the per-layer checkpoint gate and the force-computation
# autograd.grad call; `model_parf.PARFLM` is a *different*, unused base
# class, so checking it (as an earlier version of this cell did) gives a
# false pass. Two independent bugs can each cause the eval-time OOM in
# forward_gathered (all L layers' buffers alive at once during
# evaluate()'s torch.enable_grad() forward, with no outer .backward()
# ever around to free them):
#  1. FockMultiXiPARFLM._stack_forward gating the per-layer checkpoint on
#     `self.training` instead of `torch.is_grad_enabled()`.
#  2. MultiXiPARFLM._layer_step hard-coding `retain_graph=True` on the
#     force autograd.grad call instead of `retain_graph=self.training`
#     (retain_graph is only needed when create_graph=True; in eval it
#     just keeps every layer's buffers alive with no backward() call to
#     ever consume/free them).
# Fail fast here instead of discovering it ~1.5h later at the first eval
# call. If this cell is re-run in a kernel that already imported these
# modules before a later `git fetch/reset`, it will (correctly) still
# fail -- sys.modules caching means only a runtime restart clears it.
import inspect as _inspect
_stack_fwd_src = _inspect.getsource(FockMultiXiPARFLM._stack_forward)
_layer_step_src = _inspect.getsource(model_parf_multixi.MultiXiPARFLM._layer_step)
assert 'torch.is_grad_enabled()' in _stack_fwd_src, (
    'STALE MODULE IN THIS KERNEL: FockMultiXiPARFLM._stack_forward still '
    'gates per-layer checkpointing on `self.training` instead of '
    '`torch.is_grad_enabled()`. Restart the Colab runtime (Runtime > '
    'Restart runtime), re-run the setup cell so it fetches the latest '
    'main, then re-run from the top -- a plain re-run of this cell '
    'cannot fix an already-imported module.'
)
assert 'retain_graph=self.training' in _layer_step_src, (
    'STALE MODULE IN THIS KERNEL: MultiXiPARFLM._layer_step still '
    'hard-codes `retain_graph=True` on the force autograd.grad call '
    '(should be `retain_graph=self.training`). Restart the Colab '
    'runtime and re-run from the top.'
)
print('Eval-time-OOM fix verified present in this kernel '
      '(FockMultiXiPARFLM checkpoint gate + MultiXiPARFLM retain_graph).')

# The BAOAB/CfC integrators return the outgoing velocity through
# _layer_step_ex; a stale model_fock_parf_multixi would still call
# _layer_step and silently drop the O-step, training a Verlet model under a
# BAOAB tag.
_fock_step_src = _inspect.getsource(FockMultiXiPARFLM._fock_layer_step)
assert '_layer_step_ex' in _fock_step_src, (
    'STALE MODULE IN THIS KERNEL: FockMultiXiPARFLM._fock_layer_step still '
    'calls _layer_step instead of _layer_step_ex, so the BAOAB/CfC velocity '
    'would be discarded every layer. Restart the Colab runtime and re-run '
    'from the top.')
print('Integrator plumbing verified (_fock_layer_step -> _layer_step_ex).')

LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_openwebtext.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_openwebtext.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)

print(f'Logfreq: {LOGFREQ_FILE}')

ARCH_TIERS = [
    (384, 16, 32),
    (384, 12, 16),
    (256, 16, 16),
    (256,  8, 16),
]


def make_config(d, L, n_registers):
    return FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE, d=d, max_len=1024,
        L=L, v_hidden=1024, v_depth=3, dt=1.0,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_FILE),
        logfreq_init_alpha=0.1,
        init_gamma=1.0,
        fixed_gamma=FIXED_GAMMA,
        integrator=CFG_INTEGRATOR,
        vtheta_analytic_force=CFG_VTHETA_ANALYTIC,
        langevin_T=LANGEVIN_T,
        causal_force=True,
        ln_after_step=True,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode='explicit',
        v_phi_kind=V_PHI_KIND,
        v_phi_d_type=V_PHI_D_TYPE,
        v_phi_d_angle=V_PHI_D_ANGLE,
        v_phi_eps=0.1,
        v_phi_phi_hidden=128,
        v_phi_theta_hidden=128,
        v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
        top_k=TOP_K,
        v_phi_n_heads=V_PHI_N_HEADS,
        use_output_bias=USE_OUTPUT_BIAS,
        tie_embeddings=TIE_EMBEDDINGS,
        score_head_hidden=32,
        gumbel_tau_init=1.0,
        gumbel_tau_min=0.3,
        gumbel_noise=True,
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        fock_version='v2',
        n_registers=n_registers,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=64,
        stack_discipline=True,
        d_k=64,
        tau_create_init=8.0,
        reverse_channel=REVERSE_CHANNEL,
        reverse_channel_stable=REVERSE_CHANNEL_STABLE,
        reverse_channel_pre_ln=REVERSE_CHANNEL_PRE_LN,
        reverse_channel_soft_norm=REVERSE_CHANNEL_SOFT_NORM,
        reverse_channel_warmup_steps=REVERSE_CHANNEL_WARMUP_STEPS,
        reverse_channel_per_layer=REVERSE_CHANNEL_PER_LAYER,
        per_register_tau=True,
        per_register_keys=True,
        ortho_register_init=True,
        register_repulsion=REGISTER_REPULSION,
        register_repulsion_coeff=REGISTER_REPULSION_COEFF,
        register_repulsion_kind=REGISTER_REPULSION_KIND,
        prefix_causal_registers=True,
    )


model = None
model_cfg = None
for d, L, M in ARCH_TIERS:
    try:
        cfg = make_config(d, L, M)
        mdl = FockMultiXiPARFLM(cfg).to(DEVICE)
        n_v_theta_mlp = sum(p.numel() for p in mdl.V_theta.parameters())

        _init_log_prec = -math.log(d)
        _prec_max = 2.0 / d
        mdl.V_theta = AnisotropicDepthConditionedGaussianVTheta(
            d=d,
            K=V_THETA_WELLS_PER_HEAD,
            n_ctx=V_THETA_N_HEADS,
            n_layers=cfg.L,
            rank=ANISO_RANK,
            w_scale=W_SCALE,
            init_log_precision=_init_log_prec,
            precision_max=_prec_max,
            code_init_std=V_THETA_DEPTH_CODE_INIT_STD,
        ).to(DEVICE)
        install_aniso_depth_routing(mdl)

        n = mdl.num_params()
        n_v_theta = sum(p.numel() for p in mdl.V_theta.parameters())
        print(f'Trying d={d} L={L} M={M} -> {n:,} params '
              f'(V_theta {n_v_theta_mlp:,} MLP -> {n_v_theta:,} Aniso-Gaussian)')

        if DEVICE == 'cuda':
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, 2, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = mdl(_x, _y)
            _loss.backward()
            mdl.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            print(f'OOM probe passed (batch=2)')
        model = mdl
        model_cfg = cfg
        break
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM at d={d} L={L} M={M} -- trying next tier ...')
            del mdl
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue
        raise

if model is None:
    raise RuntimeError('All architecture tiers OOMed.')

if USE_OUTPUT_BIAS:
    _ob_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
    model.init_output_bias_from_logfreq(_ob_counts)
    print(f'Output bias <- log-unigram-freq  '
          f'(b range [{model.out_bias.min().item():.2f}, '
          f'{model.out_bias.max().item():.2f}])')

# -- Auto batch size, with the effective batch held fixed --
# The probe deliberately reserves headroom the bare forward+backward
# below does not use: AdamW allocates exp_avg + exp_avg_sq (2x params in
# fp32) lazily at the *first* optimizer step, i.e. after this probe has
# already passed.  Probing at bs and then training at bs is how a run
# survives the probe and OOMs on step 1.
BATCH_SIZE = 4
if DEVICE == 'cuda':
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    _probe_sizes = [16, 12, 8, 6, 4] if _vram_gb >= 70 else [12, 8, 6, 4]
    _reserve = 2.5 * sum(p.numel() for p in model.parameters()) * 4 / 1e9
    print(f'Batch probe: reserving {_reserve:.1f} GB for optimizer state '
          f'+ slack on a {_vram_gb:.0f} GB device')
    for bs in _probe_sizes:
        try:
            _rng = np.random.default_rng(42)
            _xb, _yb = get_batch(train_ids, bs, BLOCK_SIZE, _rng)
            _x = torch.from_numpy(_xb).to(DEVICE)
            _y = torch.from_numpy(_yb).to(DEVICE)
            _, _loss = model(_x, _y)
            _loss.backward()
            _peak = torch.cuda.max_memory_allocated() / 1e9
            model.zero_grad(set_to_none=True)
            del _x, _y, _xb, _yb, _loss
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            if _peak + _reserve > 0.92 * _vram_gb:
                print(f'  bs={bs}: peak {_peak:.1f} GB + reserve would not '
                      f'leave room for the optimizer -- trying next tier')
                continue
            BATCH_SIZE = bs
            print(f'  bs={bs}: peak {_peak:.1f} GB  OK')
            break
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower():
                raise
            print(f'  bs={bs}: OOM -- trying next tier')
            model.zero_grad(set_to_none=True)
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
            continue

# Spend whatever the device could not fit per-step on accumulation, so
# EFFECTIVE_BATCH (and hence gradient noise) matches the Verlet run.
GRAD_ACCUM = max(GRAD_ACCUM, -(-TARGET_EFFECTIVE_BATCH // BATCH_SIZE))
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
print(f'Auto batch: {BATCH_SIZE} x accum={GRAD_ACCUM} '
      f'(eff={EFFECTIVE_BATCH}, target={TARGET_EFFECTIVE_BATCH})')
if EFFECTIVE_BATCH != TARGET_EFFECTIVE_BATCH:
    print(f'  NOTE: effective batch {EFFECTIVE_BATCH} != target '
          f'{TARGET_EFFECTIVE_BATCH}; PPL is not directly comparable to '
          f'the Verlet run at eff=32.')
n_params = model.num_params()
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())

d = model_cfg.d
print(f'\nModel: FockMultiXiPARFLM v2.1 + Anisotropic Gaussian V_theta')
print(f'  params: {n_params:,}  (V_theta: {n_v_theta:,})')
print(f'  d={d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta: aniso-gaussian  rank={ANISO_RANK}  '
      f'{V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
      f'{V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} attractors')
print(f'  fock-reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  integrator: {INTEGRATOR}  gamma={FIXED_GAMMA}  T={LANGEVIN_T:g}')
print(f'  V_phi={V_PHI_KIND} x {V_PHI_N_HEADS} head(s)  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')

In [ ]:
# == Cell 6: Training loop =============================================

LR            = 3e-4
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = int(WSD_WARMUP_FRAC * TOTAL_STEPS) if LR_SCHEDULE == 'wsd' else 4000
GRAD_CLIP     = 1.0
GRAD_CLIP_VPHI = 0.3

PER_GROUP_CLIP = True
GRAD_CLIP_OVERRIDES = {
    'V_phi': GRAD_CLIP_VPHI,
    'creation_gate': 0.3,
    'destruction_gate': 0.3,
    'reverse_channel_scale': 0.1,
    'reverse_ch': 0.1,
    'register': 0.3,
    'depth_code': 0.5,
}
WATCHDOG_EXCLUDE_GROUPS = {'override:reverse_channel_scale', 'override:reverse_ch'}

GRAD_SPIKE_DEBUG     = True
GRAD_SPIKE_THRESHOLD = 100.0
GRAD_SPIKE_COOLDOWN  = 0
EVAL_INTERVAL = 500
EVAL_ITERS    = 40
LOG_INTERVAL  = 50
CAUSAL_PROBE_INTERVAL = 10_000
TRAINED_LEAK_PROBE_INTERVAL = 10_000
TRAINED_LEAK_PROBE_K = 256
TRAINED_LEAK_PROBE_PAIRS = 2

if WSD_LR_FLOOR is None:
    WSD_LR_FLOOR = LR * 0.05

GRAD_NORM_EMA_ALPHA = 0.05
GRAD_NORM_EMA_THRESHOLD = 50.0
GRAD_NORM_EMA_PATIENCE = 200

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)


def lr_schedule(step):
    if LR_SCHEDULE == 'wsd':
        warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
        stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
        if step < warmup_end:
            return LR * (step + 1) / max(warmup_end, 1)
        elif step < stable_end:
            return LR
        else:
            decay_steps = TOTAL_STEPS - stable_end
            progress = (step - stable_end) / max(decay_steps, 1)
            cos_decay = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
            return WSD_LR_FLOOR + (LR - WSD_LR_FLOOR) * cos_decay
    else:
        if step < WARMUP_STEPS:
            return LR * (step + 1) / WARMUP_STEPS
        progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
        return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def fock_coupling_reg(mdl, lam, eps):
    alphas = mdl.xi_module.alpha
    return -lam * torch.log(alphas + eps).sum()


def forward_with_vreg(x, targets, lambda_v, lambda_fock, fock_eps):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    fock_reg_value = torch.tensor(0.0, device=x.device)
    loss = loss_ntp

    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_reg_value = (V_vals.float() ** 2).mean()
        loss = loss + lambda_v * v_reg_value

    if lambda_fock > 0:
        fock_reg_value = fock_coupling_reg(model, lambda_fock, fock_eps)
        loss = loss + fock_reg_value

    return loss, loss_ntp, v_reg_value, fock_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    _mem_pre_gb = torch.cuda.memory_allocated() / 1e9
    torch.cuda.reset_peak_memory_stats()
    _mem_per_iter = []
    for _i in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
        del loss, x, y
        if _i % 10 == 0 or _i == EVAL_ITERS - 1:
            _mem_per_iter.append(torch.cuda.memory_allocated() / 1e9)
    _mem_post_gb = torch.cuda.memory_allocated() / 1e9
    _mem_peak_gb = torch.cuda.max_memory_allocated() / 1e9
    _mem_trend = ', '.join(f'{m:.2f}' for m in _mem_per_iter)
    print(f'    [evaluate] mem before={_mem_pre_gb:.2f}GB  '
          f'after={_mem_post_gb:.2f}GB  peak_during={_mem_peak_gb:.2f}GB  '
          f'trend(every 10 iters)=[{_mem_trend}]GB')
    model.train()
    return float(np.mean(losses))


def run_causal_probe(step_num):
    import math as _math
    # Use the same V_theta family and the same integrator as the run, so
    # this probe certifies prefix-causality for what is actually training.
    from model_aniso_gaussian_vtheta import (
        AnisotropicDepthConditionedGaussianVTheta as _DCMCGVT,
        install_aniso_depth_routing as _idr,
    )
    _PROBE_VOCAB, _PROBE_D, _PROBE_L = 101, 32, 4
    _PROBE_T, _PROBE_M, _PROBE_XI = 48, 8, 3
    _PROBE_WELLS = 4

    _logfreq_probe = Path('/tmp/causal_probe_logfreq.npy')
    np.save(_logfreq_probe, np.full(_PROBE_VOCAB, 5.0, dtype=np.float32))

    _probe_cfg = FockMultiXiPARFConfig(
        vocab_size=_PROBE_VOCAB, d=_PROBE_D, max_len=64, L=_PROBE_L,
        v_hidden=64, v_depth=3, dt=1.0,
        mass_mode='logfreq', logfreq_path=str(_logfreq_probe),
        logfreq_init_alpha=0.1, init_gamma=1.0, fixed_gamma=0.30,
        causal_force=True, ln_after_step=True,
        xi_channels=_PROBE_XI, xi_alpha_inits=[0.5, 0.9, 0.99],
        xi_learnable=True, xi_alpha_init_mode='explicit',
        v_phi_kind='structural_competitive',
        v_phi_d_type=8, v_phi_d_angle=4, v_phi_eps=0.1,
        v_phi_phi_hidden=16, v_phi_theta_hidden=16, v_phi_mlp_hidden=16,
        top_k=8, v_phi_n_heads=2,
        use_output_bias=True, tie_embeddings=False,
        score_head_hidden=8,
        gumbel_tau_init=1.0, gumbel_tau_min=0.3, gumbel_noise=True,
        use_gathered_v_phi=True, use_layer_checkpoint=False,
        ln_before_distance=True, per_layer_v_phi_scale=True,
        fock_version='v2', n_registers=_PROBE_M,
        register_salience_decay=0.5, register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True,
        d_k=16, tau_create_init=8.0,
        reverse_channel=True, reverse_channel_stable=True,
        reverse_channel_pre_ln=True, reverse_channel_soft_norm=True,
        reverse_channel_warmup_steps=4000, reverse_channel_per_layer=True,
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True, register_repulsion=False,
        prefix_causal_registers=True,
        integrator=CFG_INTEGRATOR,
        vtheta_analytic_force=CFG_VTHETA_ANALYTIC,
        langevin_T=0.0,          # noise would swamp the leak signal
    )
    torch.manual_seed(1234)
    _probe_model = FockMultiXiPARFLM(_probe_cfg)
    _probe_model.V_theta = _DCMCGVT(
        d=_PROBE_D, K=_PROBE_WELLS, n_ctx=_PROBE_XI, n_layers=_PROBE_L,
        rank=2, w_scale=1.0, init_log_precision=-_math.log(_PROBE_D),
        precision_max=2.0/_PROBE_D, code_init_std=0.02,
    )
    _idr(_probe_model)
    _probe_model.double().eval()

    with torch.no_grad():
        _probe_model.reverse_channel_scale.fill_(1.0)
        _probe_model.reverse_warmup_step.fill_(4000)

    _t_p = _PROBE_T // 2
    _prng = np.random.default_rng(7)
    _x1 = torch.from_numpy(_prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T))).long()
    _x2 = _x1.clone()
    _x2[:, _t_p:] = torch.from_numpy(
        _prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T - _t_p))).long()

    with torch.enable_grad():
        _la = _probe_model(_x1)[0].detach()
        _lb = _probe_model(_x2)[0].detach()
    _max_delta = float((_la[:, :_t_p] - _lb[:, :_t_p]).abs().max().item())

    _probe_model.train()
    torch.manual_seed(99)
    with torch.enable_grad():
        _lta = _probe_model(_x1)[0].detach()
    torch.manual_seed(99)
    with torch.enable_grad():
        _ltb = _probe_model(_x2)[0].detach()
    _max_delta = max(_max_delta,
                     float((_lta[:, :_t_p] - _ltb[:, :_t_p]).abs().max().item()))

    _passed = (_max_delta == 0.0)
    del _probe_model, _la, _lb, _lta, _ltb, _x1, _x2
    gc.collect()

    status = 'PASS' if _passed else '*** FAIL ***'
    print(f'\n[causal probe] step {step_num:,}  max|dlogit|={_max_delta:.3e}  [{status}]')
    if not _passed:
        print('[causal probe] WARNING: nonzero future sensitivity detected!')
    return _passed, _max_delta


def run_trained_leak_probe(step_num):
    _debug_dir = str(CA_DIR / 'scaleup' / 'debug')
    if _debug_dir not in sys.path:
        sys.path.insert(0, _debug_dir)
    from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

    print(f'\n{"="*64}')
    print(f'[trained leak probe] step {step_num:,} -- running on live model')
    print(f'{"="*64}')

    probe_res = probe_trained_leak(
        model, val_ids, device=DEVICE, context=BLOCK_SIZE,
        n_pairs=TRAINED_LEAK_PROBE_PAIRS, use_float64=False)
    honest_res = honest_ppl_test(
        model, val_ids, k=TRAINED_LEAK_PROBE_K,
        context=BLOCK_SIZE, batch=BATCH_SIZE, device=DEVICE)
    model.train()

    result = {
        'step': step_num,
        'probe_max_dlogit_past': probe_res['max_dlogit_past'],
        'probe_mean_dnll_past_nats': round(probe_res['mean_dnll_past'], 6),
        'probe_gate_zero_control': probe_res['gate_zero_control'],
        'honest_k': honest_res['k'],
        'ppl_mid_window_standard': round(honest_res['ppl_mid_window'], 4),
        'ppl_last_pos_leak_free': round(honest_res['ppl_last_pos'], 4),
        'paired_diff_nats': round(honest_res['paired_diff_nats'], 6),
        'paired_diff_se': round(honest_res['paired_diff_se'], 6),
    }

    _leak_status = 'CLEAN' if result['paired_diff_nats'] < 0.1 else 'LEAK DETECTED'
    print(f'\n[trained leak probe] step {step_num:,}  '
          f'honest_PPL={result["ppl_last_pos_leak_free"]:.2f}  '
          f'standard_PPL={result["ppl_mid_window_standard"]:.2f}  '
          f'diff={result["paired_diff_nats"]:+.4f} nats  [{_leak_status}]')
    return result


def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'grad_accum': GRAD_ACCUM, 'effective_batch': EFFECTIVE_BATCH,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
            'grad_clip_vphi': GRAD_CLIP_VPHI,
            'optimizer': OPTIMIZER, 'grad_centralization': GRAD_CENTRALIZATION,
            'lambda_v': LAMBDA_V, 'v_theta_variant': V_THETA_VARIANT,
            'lr_schedule': LR_SCHEDULE,
            'v_theta_n_heads': V_THETA_N_HEADS,
            'v_theta_wells_per_head': V_THETA_WELLS_PER_HEAD,
            'v_theta_depth_condition': V_THETA_DEPTH_CONDITION,
            'v_theta_depth_code_init_std': V_THETA_DEPTH_CODE_INIT_STD,
            'aniso_rank': ANISO_RANK,
            'lambda_fock_reg': LAMBDA_FOCK_REG,
            'integrator': INTEGRATOR,
            'cfg_integrator': CFG_INTEGRATOR,
            'vtheta_analytic_force': CFG_VTHETA_ANALYTIC,
            'langevin_T': LANGEVIN_T,
        },
        'step': step_num,
        'val_loss': val_loss_val,
        'val_ppl': math.exp(val_loss_val),
        'gamma': model.gamma.item(),
        'xi_alphas': model.xi_alpha_values(),
        'variant': (f'fock_parf_multixi_v2.1_aniso_gaussian_'
                f'dcvt{V_THETA_N_HEADS}_{INTEGRATOR}'),
        'corpus': 'openwebtext',
        'phase': 7,
        'seed': SEED,
    }
    fname = f'{CKPT_PREFIX}_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    for _attempt in range(2):
        try:
            torch.save(ckpt, path)
            break
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error saving checkpoint; remounting... ({_e})')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}')
                    print(f'[WARN] Checkpoint NOT saved: {path}')
                    return None
            else:
                print(f'[WARN] Checkpoint save failed: {_e}')
                return None
    print(f'  Checkpoint saved: {path}  (PPL={math.exp(val_loss_val):.2f})')
    if '_best' in tag_suffix:
        canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        import shutil
        shutil.copy2(path, canonical)
        print(f'  Canonical best: {canonical}')
    return path


# -- Optimizer --
_trainable = [p for p in model.parameters() if p.requires_grad]
if OPTIMIZER == 'adamw':
    optim = torch.optim.AdamW(_trainable, lr=LR,
                              weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lamb':
    try:
        import torch_optimizer
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch_optimizer'])
        import torch_optimizer
    optim = torch_optimizer.Lamb(_trainable, lr=LR,
                                weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lion':
    try:
        from lion_pytorch import Lion
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lion-pytorch'])
        from lion_pytorch import Lion
    optim = Lion(_trainable, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.99))
else:
    raise ValueError(f'Unknown OPTIMIZER={OPTIMIZER!r}; choose adamw / lamb / lion')
print(f'Optimizer: {type(optim).__name__}')

# -- Resume --
if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step:,}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'], strict=False)
    if (REVERSE_CHANNEL and REVERSE_CHANNEL_STABLE and REVERSE_CHANNEL_RESET_SCALE
            and getattr(model, 'reverse_channel_scale', None) is not None):
        with torch.no_grad():
            model.reverse_channel_scale.zero_()
            if hasattr(model, 'reverse_warmup_step'):
                model.reverse_warmup_step.zero_()
        print('  [E5c] reverse_channel_scale re-zeroed + warmup reset')
    if 'optimizer_state_dict' in ckpt_data:
        try:
            optim.load_state_dict(ckpt_data['optimizer_state_dict'])
            print('  Optimizer state restored.')
        except (ValueError, KeyError) as e:
            print(f'  [info] Optimizer state incompatible, starting fresh: {e}')
    prev_ppl = ckpt_data.get('val_ppl', float('nan'))
    print(f'  Model loaded. Previous PPL: {prev_ppl:.2f}')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# -- Training state --
log_path = RESULTS_DIR / 'training_log.jsonl'
_log_fh = [log_path.open('a')]
_log_write_count = [0]
# Google Drive's FUSE mount buffers writes locally and only reliably syncs
# them to Drive on file-descriptor close. A long training run keeps a single
# handle open for its whole (up to 24h) session, so if Colab kills the
# runtime abruptly (session timeout, disconnect, OOM) any writes since the
# last close can be silently lost even though flush() succeeded locally.
# Forcing an fsync + periodic close/reopen bounds how much log history can
# be lost to roughly _LOG_REOPEN_EVERY * LOG_INTERVAL steps.
_LOG_REOPEN_EVERY = 10


def _log_write(record_str):
    for _attempt in range(2):
        try:
            _log_fh[0].write(record_str)
            _log_fh[0].flush()
            try:
                os.fsync(_log_fh[0].fileno())
            except OSError:
                pass  # fsync isn't guaranteed to be meaningful on FUSE mounts
            _log_write_count[0] += 1
            if _log_write_count[0] % _LOG_REOPEN_EVERY == 0:
                _log_fh[0].close()
                _log_fh[0] = log_path.open('a')
            return
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error on log write; remounting...')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                    try:
                        _log_fh[0].close()
                    except Exception:
                        pass
                    _log_fh[0] = log_path.open('a')
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}; log record lost.')
                    return
            else:
                print(f'[WARN] Log write failed (attempt {_attempt+1}): {_e}')
                return


t0 = time.time()
model.train()
run_ntp = 0.0
run_vreg = 0.0
run_fock_reg = 0.0
n_run = 0
n_skipped = 0

best_val_ppl = float('inf')
_best_ckpt_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'

if not _best_ckpt_path.exists():
    _step_bests = sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt'))
    if _step_bests:
        _best_ckpt_path = _step_bests[-1]
        import shutil
        _canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        shutil.copy2(_best_ckpt_path, _canonical)
        _best_ckpt_path = _canonical

if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored running best PPL: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] {e}')

_grad_norm_ema = 0.0
_grad_norm_above_thresh = 0


def _reload_best():
    if not _best_ckpt_path.exists():
        return resume_step
    ckpt = torch.load(_best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    try:
        optim.load_state_dict(ckpt['optimizer_state_dict'])
    except (ValueError, KeyError):
        pass
    s = ckpt.get('step', 0)
    p = ckpt.get('val_ppl', float('nan'))
    del ckpt
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'[watchdog] Reloaded best: step {s:,} PPL {p:.2f}')
    return s


steps_this_session = 0

# -- Schedule summary --
if LR_SCHEDULE == 'wsd':
    _warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
    _stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
    _sched_str = (f'WSD: warmup 0->{_warmup_end:,}, stable {_warmup_end:,}->{_stable_end:,}, '
                  f'decay {_stable_end:,}->{TOTAL_STEPS:,}, floor={WSD_LR_FLOOR:.2e}')
else:
    _sched_str = f'cosine: warmup {WARMUP_STEPS:,} steps'

print(f'\n{"="*60}')
print(f'CfC/BAOAB [{INTEGRATOR}]: steps {resume_step+1:,} -> {TOTAL_STEPS:,}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  block={BLOCK_SIZE}  lr={LR}  grad_clip={GRAD_CLIP}')
print(f'  schedule: {_sched_str}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  fock-reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  integrator: {INTEGRATOR}  (cfg.integrator={model_cfg.integrator}, '
      f'analytic V_theta force={model_cfg.vtheta_analytic_force}, '
      f'thermostat T={model_cfg.langevin_T:g})')
print(f'  watchdog: threshold={GRAD_NORM_EMA_THRESHOLD} patience={GRAD_NORM_EMA_PATIENCE}')
print(f'  per-group clip: default={GRAD_CLIP}  overrides={GRAD_CLIP_OVERRIDES}')
if REVERSE_CHANNEL:
    _rev_mode = ('stable (QK-norm + '
                 + ('soft-norm' if REVERSE_CHANNEL_SOFT_NORM else 'RMS-norm')
                 + (' + pre-LN' if REVERSE_CHANNEL_PRE_LN else '') + ')')
    print(f'  reverse channel: {_rev_mode}  warmup={REVERSE_CHANNEL_WARMUP_STEPS} forwards')
else:
    print('  reverse channel: OFF')
print(f'{"="*60}\n')


def _assign_clip_group(pname):
    low = pname.lower()
    for sub, thr in GRAD_CLIP_OVERRIDES.items():
        if sub.lower() in low:
            return f'override:{sub}', thr
    return pname.split('.', 1)[0], GRAD_CLIP


def per_group_grad_norms(mdl):
    groups = {}
    for n, p in mdl.named_parameters():
        if not p.requires_grad or p.grad is None:
            continue
        key, _ = _assign_clip_group(n)
        groups.setdefault(key, []).append(p)
    out = {}
    for key, ps in groups.items():
        sq = 0.0
        for p in ps:
            sq += float(p.grad.detach().norm()) ** 2
        out[key] = sq ** 0.5
    return out


def clip_grads_per_group(mdl):
    groups, thr = {}, {}
    _dev = None
    for n, p in mdl.named_parameters():
        if not p.requires_grad or p.grad is None:
            continue
        if _dev is None:
            _dev = p.grad.device
        key, mx = _assign_clip_group(n)
        groups.setdefault(key, []).append(p)
        thr[key] = mx
    total_sq = torch.zeros((), device=_dev) if _dev is not None else torch.zeros(())
    per_group = {}
    for key, ps in groups.items():
        gn = nn.utils.clip_grad_norm_(ps, thr[key])
        per_group[key] = float(gn)
        if key not in WATCHDOG_EXCLUDE_GROUPS:
            total_sq = total_sq + gn.detach() ** 2
    return total_sq.sqrt(), per_group


_last_pg_norms = {}
_last_spike_step = -10**9
for step in range(resume_step, TOTAL_STEPS):
    lr_now = lr_schedule(step)
    for g in optim.param_groups:
        g['lr'] = lr_now

    optim.zero_grad(set_to_none=True)
    accum_ntp = 0.0
    accum_vreg = 0.0
    accum_fock_reg = 0.0
    accum_rep = 0.0
    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        loss, loss_ntp, v_reg, fock_reg = forward_with_vreg(
            x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
        if REGISTER_REPULSION:
            _rep = model.pop_repulsion_loss()
            loss = loss + _rep
            accum_rep += float(_rep.detach()) / GRAD_ACCUM
        (loss / GRAD_ACCUM).backward()
        accum_ntp      += loss_ntp.item()       / GRAD_ACCUM
        accum_vreg     += float(v_reg.detach()) / GRAD_ACCUM
        accum_fock_reg += float(fock_reg.detach()) / GRAD_ACCUM

    if GRAD_CENTRALIZATION:
        for p in model.parameters():
            if p.grad is not None and p.grad.dim() >= 2:
                p.grad.sub_(p.grad.mean(dim=tuple(range(1, p.grad.dim())), keepdim=True))

    if PER_GROUP_CLIP:
        grad_norm, _last_pg_norms = clip_grads_per_group(model)
    else:
        _last_pg_norms = per_group_grad_norms(model) if GRAD_SPIKE_DEBUG else {}
        if model.V_phi is not None:
            nn.utils.clip_grad_norm_(model.V_phi.parameters(), GRAD_CLIP_VPHI)
        grad_norm = nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)

    if GRAD_SPIKE_DEBUG:
        _tot_preclip = float(grad_norm)
        if (_tot_preclip > GRAD_SPIKE_THRESHOLD
                and (step - _last_spike_step) >= GRAD_SPIKE_COOLDOWN):
            _last_spike_step = step
            if _last_pg_norms:
                _top = sorted(_last_pg_norms.items(),
                              key=lambda kv: kv[1], reverse=True)[:8]
                _brk = '  '.join(f'{k}={v:.1f}' for k, v in _top)
            else:
                _brk = '(enable PER_GROUP_CLIP for breakdown)'
            print(f'\n[spike] step {step+1}: pre-clip total grad={_tot_preclip:.1f}  '
                  f'ntp={accum_ntp:.3f}  v_reg={accum_vreg:.4f}  fock_reg={accum_fock_reg:.4f}')
            print(f'[spike]   top groups: {_brk}')
            _log_write(json.dumps({
                'step': step + 1, 'event': 'grad_spike',
                'pre_clip_grad_norm': round(_tot_preclip, 2),
                'ntp': round(accum_ntp, 4), 'v_reg': round(accum_vreg, 4),
                'fock_reg': round(accum_fock_reg, 4),
                'top_groups': {k: round(v, 2) for k, v in _top} if _last_pg_norms else {},
            }) + '\n')

    if torch.isfinite(grad_norm) and math.isfinite(accum_ntp):
        optim.step()
        for bank in model.V_theta.banks:
            if hasattr(bank, 'clamp_params'):
                bank.clamp_params()
    else:
        n_skipped += 1
        optim.zero_grad(set_to_none=True)

    # -- Watchdog --
    _raw_gn = float(grad_norm)
    _grad_norm_ema = (1 - GRAD_NORM_EMA_ALPHA) * _grad_norm_ema + GRAD_NORM_EMA_ALPHA * _raw_gn
    if _grad_norm_ema > GRAD_NORM_EMA_THRESHOLD:
        _grad_norm_above_thresh += 1
    else:
        _grad_norm_above_thresh = 0

    if _grad_norm_above_thresh >= GRAD_NORM_EMA_PATIENCE:
        print(f'\n[watchdog] EMA grad_norm={_grad_norm_ema:.1f} > {GRAD_NORM_EMA_THRESHOLD} '
              f'for {_grad_norm_above_thresh} steps at step {step+1}.')
        if _last_pg_norms:
            _top = sorted(_last_pg_norms.items(), key=lambda kv: kv[1], reverse=True)[:5]
            print('[watchdog] top group norms (pre-clip): '
                  + ', '.join(f'{k}={v:.1f}' for k, v in _top))
        _log_write(json.dumps({
            'step': step + 1, 'event': 'watchdog_reload',
            'ema_grad_norm': round(_grad_norm_ema, 2),
            'above_thresh_steps': _grad_norm_above_thresh,
        }) + '\n')
        _reload_best()
        _grad_norm_ema = 0.0
        _grad_norm_above_thresh = 0
        n_skipped += 1

    run_ntp += accum_ntp
    run_vreg += accum_vreg
    run_fock_reg += accum_fock_reg
    n_run += 1
    steps_this_session += 1

    if (step + 1) % LOG_INTERVAL == 0:
        avg_ntp = run_ntp / n_run
        avg_vreg = run_vreg / n_run
        avg_fock_reg = run_fock_reg / n_run
        run_ntp, run_vreg, run_fock_reg, n_run = 0.0, 0.0, 0.0, 0
        elapsed = time.time() - t0
        sec_per_step = elapsed / steps_this_session
        remaining = (TOTAL_STEPS - step - 1) * sec_per_step
        alphas = model.xi_alpha_values()
        alpha_str = ','.join(f'{a:.3f}' for a in alphas)
        _top_grp = ''
        if PER_GROUP_CLIP and _last_pg_norms:
            _k, _v = max(_last_pg_norms.items(), key=lambda kv: kv[1])
            _top_grp = f'top[{_k}]={_v:.1f}  '
        _rep_str = f'rep={accum_rep:.4f}  ' if REGISTER_REPULSION else ''
        _mem_alloc_gb = torch.cuda.memory_allocated() / 1e9
        _mem_resv_gb = torch.cuda.memory_reserved() / 1e9
        _mem_peak_gb = torch.cuda.max_memory_allocated() / 1e9
        torch.cuda.reset_peak_memory_stats()
        print(
            f'step {step+1:7d}/{TOTAL_STEPS}  '
            f'ntp={avg_ntp:.4f}  v_reg={avg_vreg:.4f}  fock_reg={avg_fock_reg:.4f}  '
            f'lr={lr_now:.2e}  grad={float(grad_norm):.2f}  {_rep_str}{_top_grp}'
            f'gamma={model.gamma.item():.3f}  alpha=[{alpha_str}]  '
            f'mem_alloc={_mem_alloc_gb:.1f}GB  mem_resv={_mem_resv_gb:.1f}GB  '
            f'mem_peak={_mem_peak_gb:.1f}GB  '
            f'{elapsed:.0f}s  (~{remaining/3600:.1f}h remaining)')
        _log_write(json.dumps({
            'step': step + 1, 'train_loss': avg_ntp, 'v_reg': avg_vreg,
            'fock_reg': avg_fock_reg,
            'lr': lr_now, 'grad_norm': float(grad_norm),
            'gamma': model.gamma.item(), 'xi_alphas': alphas,
            'reg_repulsion': accum_rep,
            'mem_alloc_gb': round(_mem_alloc_gb, 3),
            'mem_reserved_gb': round(_mem_resv_gb, 3),
            'mem_peak_gb': round(_mem_peak_gb, 3),
            'elapsed_sec': elapsed, 'sec_per_step': sec_per_step,
        }) + '\n')

    if (step + 1) % EVAL_INTERVAL == 0:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        is_best = val_ppl < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl
        elapsed = time.time() - t0
        marker = '*** NEW BEST ***' if is_best else ''
        print(f'>>> EVAL step {step+1:,}  val_loss={val_loss:.4f}  '
              f'val_ppl={val_ppl:.2f}  best={best_val_ppl:.2f}  '
              f'{marker}  ({elapsed:.0f}s)')
        _log_write(json.dumps({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
        }) + '\n')
        if is_best:
            save_checkpoint(step + 1, val_loss, tag_suffix='_best')

    if (step + 1) in set(CKPT_STEPS):
        if (step + 1) % EVAL_INTERVAL != 0:
            val_loss = evaluate()
            val_ppl = math.exp(val_loss)
        save_checkpoint(step + 1, val_loss)

    if CAUSAL_PROBE_INTERVAL > 0 and (step + 1) % CAUSAL_PROBE_INTERVAL == 0:
        _cp_passed, _cp_delta = run_causal_probe(step + 1)
        _log_write(json.dumps({
            'step': step + 1,
            'causal_probe_passed': _cp_passed,
            'causal_probe_max_delta': _cp_delta,
        }) + '\n')

    if TRAINED_LEAK_PROBE_INTERVAL > 0 and (step + 1) % TRAINED_LEAK_PROBE_INTERVAL == 0:
        _tlp_result = run_trained_leak_probe(step + 1)
        _log_write(json.dumps(_tlp_result) + '\n')

_log_fh[0].close()
print(f'\nTraining complete. Best PPL: {best_val_ppl:.2f}')

In [ ]:
# == Cell 6b: Stiffness diagnostic — how much is the CfC actually saving? ==
#
# The explicit (Verlet) layer step is stable only while omega*dt < 2, where
# omega = sqrt(K/m) is the local V_theta curvature seen by one coordinate of
# one token.  This probe measures the distribution of omega*dt across a real
# batch, so the instability can be observed *directly* rather than inferred
# from the gradient norm after the fact.
#
# Read it as: any mass above omega*dt = 2 is a coordinate the Verlet
# integrator is provably amplifying, and that the CfC propagator rotates
# instead.  Safe to run against any arm -- it temporarily borrows the
# harmonic linearisation even when training under 'verlet'.

import contextlib

def stiffness_report(mdl, x, dt=None):
    """Distribution of omega*dt over layers, tokens and dimensions."""
    dt = float(mdl.cfg.dt if dt is None else dt)
    if not hasattr(mdl.V_theta, 'harmonic_terms'):
        raise RuntimeError('V_theta has no harmonic_terms(); need the '
                           'anisotropic Gaussian family.')

    seen = []
    _orig = mdl.V_theta.harmonic_terms

    def _recording(xis, h):
        k_diag, s = _orig(xis, h)
        seen.append(k_diag.detach().float().flatten().cpu())
        return k_diag, s

    _saved = (mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force)
    mdl.V_theta.harmonic_terms = _recording
    mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force = 'baoab_cfc', True
    try:
        was_training = mdl.training
        mdl.eval()
        with torch.enable_grad():
            mdl(x)
    finally:
        mdl.V_theta.harmonic_terms = _orig
        mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force = _saved
        if was_training:
            mdl.train()

    k = torch.cat(seen)
    m = float(mdl.compute_mass(x).mean())
    wdt = (k.clamp(min=0) / m).sqrt() * dt
    n_total = int(wdt.numel())
    # The tails are what matter, so keep the exact max but subsample for the
    # quantiles: torch.quantile refuses inputs beyond ~16M elements, and
    # L*B*T*d reaches that at d=384, L=16 with a large auto-probed batch.
    wdt_max = float(wdt.max())
    if n_total > 4_000_000:
        idx = torch.randint(0, n_total, (4_000_000,))
        wdt_q = wdt[idx]
    else:
        wdt_q = wdt
    q = torch.tensor([0.5, 0.9, 0.99, 0.999])
    qs = torch.quantile(wdt_q.double(), q.double()).float()
    qs = torch.cat([qs, torch.tensor([wdt_max])])
    return {
        'n_samples': n_total,
        'mean_mass': m,
        'median': float(qs[0]), 'p90': float(qs[1]), 'p99': float(qs[2]),
        'p999': float(qs[3]), 'max': float(qs[4]),
        'frac_unstable': float((wdt > 2.0).float().mean()),
        'frac_marginal': float((wdt > 1.0).float().mean()),
    }


_rng_s = np.random.default_rng(0)
_xb, _ = get_batch(train_ids, min(BATCH_SIZE, 4), BLOCK_SIZE, _rng_s)
_rep = stiffness_report(model, torch.from_numpy(_xb).to(DEVICE))

print(f'omega*dt over {_rep["n_samples"]:,} (layer, token, dim) samples '
      f'  [mean mass {_rep["mean_mass"]:.3f}]')
print(f'  median {_rep["median"]:.4f}   p90 {_rep["p90"]:.4f}   '
      f'p99 {_rep["p99"]:.4f}   p99.9 {_rep["p999"]:.4f}   '
      f'max {_rep["max"]:.4f}')
print(f'  fraction with omega*dt > 1 (marginal): {_rep["frac_marginal"]:.3e}')
print(f'  fraction with omega*dt > 2 (Verlet-unstable): '
      f'{_rep["frac_unstable"]:.3e}')
if _rep['max'] > 2.0:
    print('  => the explicit step is UNSTABLE on some coordinates right now; '
          'these are exactly what baoab_cfc integrates exactly instead.')
else:
    print('  => no coordinate exceeds the explicit stability bound at this '
          'checkpoint (re-run later in training: wells sharpen over time).')


## Fock v2.1 component diagnosticsStandalone probe -- safe to run any time against the live model or afreshly loaded checkpoint. It answers two questions:1. **Structural health** -- is each Fock piece being *used well*?2. **PPL attribution** -- how much does each piece actually *buy*?

In [ ]:
# == Cell 7: Component diagnostics =====================================
import torch
_bp = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
_bd = torch.load(_bp, map_location=DEVICE, weights_only=False)
model.load_state_dict(_bd['model_state_dict'], strict=False)
model.eval()
print(f"Probe target -> {_bp.name}  step {_bd.get('step')}  PPL {_bd.get('val_ppl'):.2f}")
del _bd
import gc; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

import gc, math, sys, torch, numpy as np
for _a in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _a): setattr(sys, _a, None)
model.zero_grad(set_to_none=True)
try: optim.zero_grad(set_to_none=True)
except Exception: pass
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU free {_free/1e9:.1f} / {_total/1e9:.1f} GB before probe')

PROBE_BS = 2
_rng = np.random.default_rng(1234)
def _mk(n, bs):
    return [(torch.from_numpy(a).to(DEVICE), torch.from_numpy(b).to(DEVICE))
            for a, b in (get_batch(val_ids, bs, BLOCK_SIZE, _rng) for _ in range(n))]
def _eval_on(batches):
    model.eval(); losses = []
    for x, y in batches:
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(float(loss.item()))
        del loss
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return float(np.mean(losses))

# --- 1. structural health ---
model.eval(); model.set_fock_capture(True)
_hx, _hy = _mk(1, PROBE_BS)[0]
with torch.enable_grad():
    _out = model(_hx, _hy)
del _out
rep = model.fock_component_report()
_cols = ['layer','active_frac','reg_cos_sim','create_entropy','create_alpha_max',
         'rev_entropy','rev_scale','qforce_ratio','destroy_mean']
print('='*72); print('Fock v2.1 STRUCTURAL HEALTH'); print('='*72)
print('  '.join(f'{c[:10]:>10}' for c in _cols))
for dd in rep['per_layer']:
    print('  '.join(f'{str(dd.get(c)):>10}' if isinstance(dd.get(c),(bool,type(None)))
                    else f'{float(dd.get(c)):>10.3f}' for c in _cols))
print('-'*72); print('summary:', {k: round(v,3) for k,v in rep['summary'].items()})
for f in rep.get('flags', []): print('  * '+f)
del _hx, _hy, rep; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

# --- 2. PPL attribution ---
_pb = _mk(40, PROBE_BS)
base = _eval_on(_pb); base_ppl = math.exp(base); rows = [('full model', base)]
if getattr(model, 'reverse_channel_scale', None) is not None:
    _s = model.reverse_channel_scale.detach().clone()
    with torch.no_grad(): model.reverse_channel_scale.zero_()
    rows.append(('  - reverse channel', _eval_on(_pb)))
    with torch.no_grad(): model.reverse_channel_scale.copy_(_s)
_thr = model.cfg.register_salience_threshold
try:
    model.cfg.register_salience_threshold = 1e9
    rows.append(('  - registers (all)', _eval_on(_pb)))
finally:
    model.cfg.register_salience_threshold = _thr
print('\n'+'='*72)
print(f"{'arm':<22}{'loss':>10}{'ppl':>10}{'dPPL':>10}")
for n, l in rows:
    p = math.exp(l); print(f'{n:<22}{l:>10.4f}{p:>10.2f}{p-base_ppl:>+10.2f}')

In [ ]:
# == Cell 8: Training curve ============================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

eval_entries = []
alpha_entries = []
if log_path.exists():
    with open(log_path) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e and 'event' not in e:
                    eval_entries.append(e)
                if 'xi_alphas' in e and 'event' not in e:
                    alpha_entries.append(e)
            except Exception:
                pass

if eval_entries:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    steps_arr = [e['step'] for e in eval_entries]
    ppls = [e['val_ppl'] for e in eval_entries]
    ax.plot(steps_arr, ppls, 'o-',
            label=f'Aniso-Gaussian r={ANISO_RANK} + fock-reg (OWT d=384)',
            linewidth=1.5, color='#C62828')
    ax.axhline(y=9.14, color='green', linestyle='--', alpha=0.7,
               label='Aniso-Gaussian TinyStories best (9.14)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Val PPL')
    ax.set_title(f'Aniso-Gaussian + Fock-Reg -- OpenWebText d=384')
    ax.legend()
    ax.grid(True, alpha=0.3)

    if alpha_entries:
        ax = axes[1]
        a_steps = [e['step'] for e in alpha_entries]
        n_ch = len(alpha_entries[0]['xi_alphas'])
        for k in range(n_ch):
            ax.plot(a_steps, [e['xi_alphas'][k] for e in alpha_entries],
                    'o-', label=f'alpha_{k+1}', markersize=2, linewidth=1.5)
        ax.set_xlabel('Step')
        ax.set_ylabel('alpha_k')
        ax.set_title('Fock coupling strengths (alpha_k)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    fig.savefig(RESULTS_DIR / 'training_curve_aniso_gaussian_owt.png', dpi=150)
    plt.show()
    print(f'Saved: {RESULTS_DIR / "training_curve_aniso_gaussian_owt.png"}')
else:
    print('No eval data to plot.')